In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import joblib

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv(
    r"E:\eeg-analysis\eeg-analysis\cognitive_state_discrimination_dataset - cognitive_state_discrimination_dataset.csv"
)

print("="*60)
print("DATA INFO")
print("="*60)
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

# ============================================================
# 2. PREPARE FEATURES & TARGET
# ============================================================
# Y = Target Label
y = df["Target_Label"]
print(df["Target_Label"])

# X = 64 Channels + Time_Stamp
# Exclude hanya: Participant_ID, Target_Label, Task
exclude_cols = ["Participant_ID", "Target_Label", "Task"]
X = df.drop(columns=exclude_cols)

print("\n" + "="*60)
print("FEATURES (X)")
print("="*60)
print(f"Shape: {X.shape}")
print(f"Columns: {X.columns.tolist()}")
print(f"\nSample data:\n{X.head()}")

# ============================================================
# 3. HANDLE CATEGORICAL (jika Time_Stamp string)
# ============================================================
categorical_cols = X.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    print(f"\nFound categorical columns: {list(categorical_cols)}")
    print("Applying Label Encoding...")
    
    X_encoded = X.copy()
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X_encoded[col] = le.fit_transform(X[col])
        label_encoders[col] = le
    X = X_encoded

# ============================================================
# 4. CHECK TARGET DISTRIBUTION
# ============================================================
print("\n" + "="*60)
print("TARGET DISTRIBUTION (Y)")
print("="*60)
print(y.value_counts())
print(f"\nClass balance:\n{y.value_counts(normalize=True)}")

# ============================================================
# 5. CROSS-VALIDATION
# ============================================================
print("\n" + "="*60)
print("RUNNING 5-FOLD CROSS-VALIDATION")
print("="*60)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

cv_results = cross_validate(
    rf,
    X,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro"
    },
    return_train_score=True,
    verbose=1
)

# ============================================================
# 6. DISPLAY RESULTS
# ============================================================
print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)

metrics = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

for metric in metrics:
    train_scores = cv_results[f"train_{metric}"]
    test_scores = cv_results[f"test_{metric}"]
    
    print(f"\n{metric.upper()}:")
    print(f"  Train: {train_scores.mean():.4f} (±{train_scores.std():.4f})")
    print(f"  Test:  {test_scores.mean():.4f} (±{test_scores.std():.4f})")
    print(f"  Gap:   {train_scores.mean() - test_scores.mean():.4f}")

# ============================================================
# 7. TRAIN FINAL MODEL & SAVE
# ============================================================
print("\n" + "="*60)
print("TRAINING FINAL MODEL")
print("="*60)

rf_final = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_final.fit(X, y)

# Save model
joblib.dump(rf_final, "rf_final_model.pkl")
print("✅ Model saved: rf_final_model.pkl")

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_final.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 20 Most Important Features:")
print(feature_importance.head(20))

# ============================================================
# 8. SUMMARY
# ============================================================
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Features used: {X.shape[1]} (64 channels + Time_Stamp)")
print(f"Target classes: {y.nunique()} classes")
print(f"\nTest Accuracy:  {cv_results['test_accuracy'].mean():.4f} (±{cv_results['test_accuracy'].std():.4f})")
print(f"Test F1-Score:  {cv_results['test_f1_macro'].mean():.4f} (±{cv_results['test_f1_macro'].std():.4f})")
print(f"Overfitting Gap: {cv_results['train_accuracy'].mean() - cv_results['test_accuracy'].mean():.4f}")

DATA INFO
Shape: (40, 71)

Columns: ['Participant_ID', 'Task', 'Channel_1_PSD', 'Channel_2_PSD', 'Channel_3_PSD', 'Channel_4_PSD', 'Channel_5_PSD', 'Channel_6_PSD', 'Channel_7_PSD', 'Channel_8_PSD', 'Channel_9_PSD', 'Channel_10_PSD', 'Channel_11_PSD', 'Channel_12_PSD', 'Channel_13_PSD', 'Channel_14_PSD', 'Channel_15_PSD', 'Channel_16_PSD', 'Channel_17_PSD', 'Channel_18_PSD', 'Channel_19_PSD', 'Channel_20_PSD', 'Channel_21_PSD', 'Channel_22_PSD', 'Channel_23_PSD', 'Channel_24_PSD', 'Channel_25_PSD', 'Channel_26_PSD', 'Channel_27_PSD', 'Channel_28_PSD', 'Channel_29_PSD', 'Channel_30_PSD', 'Channel_31_PSD', 'Channel_32_PSD', 'Channel_33_PSD', 'Channel_34_PSD', 'Channel_35_PSD', 'Channel_36_PSD', 'Channel_37_PSD', 'Channel_38_PSD', 'Channel_39_PSD', 'Channel_40_PSD', 'Channel_41_PSD', 'Channel_42_PSD', 'Channel_43_PSD', 'Channel_44_PSD', 'Channel_45_PSD', 'Channel_46_PSD', 'Channel_47_PSD', 'Channel_48_PSD', 'Channel_49_PSD', 'Channel_50_PSD', 'Channel_51_PSD', 'Channel_52_PSD', 'Channel_5

c:\Users\Alyssa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Alyssa\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



CROSS-VALIDATION RESULTS

ACCURACY:
  Train: 1.0000 (±0.0000)
  Test:  0.7250 (±0.0500)
  Gap:   0.2750

PRECISION_MACRO:
  Train: 1.0000 (±0.0000)
  Test:  0.7283 (±0.1031)
  Gap:   0.2717

RECALL_MACRO:
  Train: 1.0000 (±0.0000)
  Test:  0.7250 (±0.0500)
  Gap:   0.2750

F1_MACRO:
  Train: 1.0000 (±0.0000)
  Test:  0.6936 (±0.0740)
  Gap:   0.3064

TRAINING FINAL MODEL
✅ Model saved: rf_final_model.pkl

Top 20 Most Important Features:
               Feature  Importance
66  ERP_Visual_Pattern    0.062744
64   ERP_Memory_Recall    0.060196
65      ERP_Arithmetic    0.057103
49      Channel_50_PSD    0.040563
54      Channel_55_PSD    0.029048
2        Channel_3_PSD    0.027287
5        Channel_6_PSD    0.027178
21      Channel_22_PSD    0.025049
57      Channel_58_PSD    0.022378
12      Channel_13_PSD    0.020387
37      Channel_38_PSD    0.020033
27      Channel_28_PSD    0.019485
63      Channel_64_PSD    0.017328
26      Channel_27_PSD    0.017183
13      Channel_14_PSD    0.01677

In [11]:
joblib.dump(rf_final, "rf_final_model.pkl")
print("✅ rf_final_model.pkl berhasil disimpan!")

✅ rf_final_model.pkl berhasil disimpan!
